# Kafka, first steps

Twenty minutes, one topic, and the four facts that the rest of the Kafka lecture
assumes you have seen: **a topic is a set of partitions**, **the key decides the
partition**, **a consumer group splits the partitions between its members**, and
**a group is a cursor, not a queue**.

Every section asks you to **predict before you run**. The predictions are the point;
the outputs only settle them. Nothing here needs Spark — this is Kafka on its own,
which is the only way to see the parts that Spark and ksqlDB hide from you.

### The client is already installed

There is no `pip` cell in this module: `docker-compose.yml` mounts a script into the
notebook image's `before-notebook.d/`, so `confluent-kafka` is installed when the
container starts. The cell below is a check. If it fails, one line on your host:

```
docker exec -it kafka-first-steps-notebook pip install confluent-kafka==2.5.0
```

In [1]:
import confluent_kafka
print("confluent-kafka", confluent_kafka.version()[0])

confluent-kafka 2.5.0


In [2]:
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import Producer, Consumer, TopicPartition
import time, collections, json, datetime

BOOTSTRAP = "kafka:9092"      # the broker, inside the docker network
TOPIC     = "elevator-doors"  # a sandbox topic with 3 partitions. The fleet's own
                              # topic, in lecture 10, is elevator-events with 6: the
                              # number is the lesson, so it gets a name of its own

def door_event(unit, seq, floor=3, direction="UP"):
    """One door event, exactly as the fleet simulator of lecture 10 puts it on the wire:
    same six fields, same order, same json.dumps. 115 bytes, every time — `seq` moves the
    timestamp by whole seconds, so the length never changes."""
    ts = datetime.datetime(2026, 10, 15, 8, 0, 0) + datetime.timedelta(seconds=seq)
    return json.dumps({"event": "DoorOpened", "unitId": unit, "ts": ts.isoformat(),
                       "car": "A", "floor": floor, "servedDir": direction})

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
print("broker reachable, topics now:", sorted(admin.list_topics(timeout=10).topics))
print("one event  :", door_event("EU-001", 0))
print("its length :", len(door_event("EU-001", 0)), "bytes")

broker reachable, topics now: ['__consumer_offsets']
one event  : {"event": "DoorOpened", "unitId": "EU-001", "ts": "2026-10-15T08:00:00", "car": "A", "floor": 3, "servedDir": "UP"}
its length : 115 bytes


---
## 1) A topic is a set of partitions

Three partitions, one replica each — one broker, so there is nowhere to put a second
copy. In production the replication factor is 3 and the replicas sit on different
brokers; everything below would read the same.

**Why three and not one?** Because a partition is the unit of both ordering and
parallelism: order is guaranteed *inside* a partition and nowhere else, and a consumer
group can never have more working members than the topic has partitions. Choosing the
partition count is choosing the maximum parallelism, before a single message exists.

In [3]:
fs = admin.create_topics([NewTopic(TOPIC, num_partitions=3, replication_factor=1)])
for t, f in fs.items():
    f.result()          # raises if the creation failed
    print("created", t)

# the broker needs a moment to publish the new metadata
for _ in range(10):
    meta = admin.list_topics(timeout=10).topics
    if TOPIC in meta and len(meta[TOPIC].partitions) == 3:
        break
    time.sleep(0.5)

for pid, p in sorted(meta[TOPIC].partitions.items()):
    print(f"partition {pid}: leader=broker {p.leader}, replicas={p.replicas}")

created elevator-doors
partition 0: leader=broker 1, replicas=[1]
partition 1: leader=broker 1, replicas=[1]
partition 2: leader=broker 1, replicas=[1]


---
## 2) With no key, the client spreads them itself

Nine door events — the real one, the same JSON the simulator of lecture 10 produces — and
**no key at all**. Note what that means: `unitId` is *inside* the payload, and the broker
never looks inside. A field is not a key until you make it one.

**Predict first:** how will they be spread over the three partitions? And in which
partition will message number 7 land?

In [4]:
seen = []

def report(err, msg, i=None):
    """Delivery callback: the broker tells us where the message actually went.

    `i` is the production index, captured at produce time. The callbacks come back
    grouped by BATCH, not in production order, and nothing inside a Kafka message
    records the order it was sent in — so if we want that order, we carry it ourselves.
    """
    if err:
        print("FAILED:", err)
    else:
        seen.append((i, msg.key(), msg.value().decode(), msg.partition(), msg.offset()))

producer = Producer({"bootstrap.servers": BOOTSTRAP})

# The topic was created seconds ago, so the client has never heard of it. The first
# produce() would block on a metadata round trip, and that wait alone can outlast the
# 10 ms sticky window we are about to measure. list_topics() warms the cache without
# sending anything, which keeps the message count honest for the consumer sections.
producer.list_topics(topic=TOPIC, timeout=10)

events = [door_event("EU-001", i) for i in range(9)]   # built before, not in the loop

t0 = time.perf_counter()
for i in range(9):
    producer.produce(TOPIC, value=events[i],
                     on_delivery=lambda err, msg, i=i: report(err, msg, i))
loop_ms = (time.perf_counter() - t0) * 1000
producer.flush()

seen.sort(key=lambda r: r[0])
for i, key, val, part, off in seen:
    print(f"event {i} -> partition {part}, offset {off}")

print()
print("what went on the wire:", seen[0][2])

order = [part for i, key, val, part, off in seen]
spread = collections.Counter(order)
busiest, n_busiest = spread.most_common(1)[0]
switches = sum(1 for a, b in zip(order, order[1:]) if a != b)
print()
print("partition per message, in production order:", order)
print(f"the nine produce() calls took {loop_ms:.1f} ms   (sticky window: 10 ms)")
print(f"busiest partition: {busiest}, holding {n_busiest} of 9"
      f"   ({dict(sorted(spread.items()))})")
print(f"switches between consecutive messages: {switches} out of {len(order) - 1}")

event 0 -> partition 0, offset 0
event 1 -> partition 0, offset 1
event 2 -> partition 0, offset 2
event 3 -> partition 0, offset 3
event 4 -> partition 0, offset 4
event 5 -> partition 0, offset 5
event 6 -> partition 0, offset 6
event 7 -> partition 0, offset 7
event 8 -> partition 0, offset 8

what went on the wire: {"event": "DoorOpened", "unitId": "EU-001", "ts": "2026-10-15T08:00:00", "car": "A", "floor": 3, "servedDir": "UP"}

partition per message, in production order: [0, 0, 0, 0, 0, 0, 0, 0, 0]
the nine produce() calls took 0.1 ms   (sticky window: 10 ms)
busiest partition: 0, holding 9 of 9   ({0: 9})
switches between consecutive messages: 0 out of 8


**Count before you believe the word *round-robin*.** Strict round-robin would switch
partition on every single message — eight out of eight, three each. Read instead the two
numbers the cell printed: **how long the nine `produce()` calls took**, and **how many of
the nine are in the busiest partition**.

The client's default partitioner is `consistent_random`: the key is hashed, and a message
*without* a key is *"randomly partitioned"*. But random per **what**? Not per message.
`sticky.partitioning.linger.ms` — default **10 ms** — is a *"delay to wait to assign new
sticky partitions for each topic"*, and its stated purpose is that batching keyless
messages together is cheaper than spreading them. Nine `produce()` calls take a fraction
of a millisecond, so they fall inside one sticky window and pile into one partition.

Two honest caveats, because you may well see something else. Nine is a very small sample,
and the window is a *timer*: anything that makes the loop straddle 10 ms — a slow laptop,
a first call that still has to fetch metadata — draws a new partition mid-loop and the
concentration weakens. That is why the cell prints its own timing: if the loop came in
well under 10 ms and the messages still spread, the explanation above is the one that is
wrong, and the number on the screen wins.

**So predict again:** the same nine messages, 30 ms apart — longer than the sticky
window. Where will they go?

In [5]:
seen.clear()
for i in range(9):
    producer.produce(TOPIC, value=events[i],
                     on_delivery=lambda err, msg, i=i: report(err, msg, i))
    producer.flush()          # close the batch
    time.sleep(0.03)          # longer than sticky.partitioning.linger.ms (10 ms)

seen.sort(key=lambda r: r[0])
order = [part for i, key, val, part, off in seen]
spread = collections.Counter(order)
busiest, n_busiest = spread.most_common(1)[0]
switches = sum(1 for a, b in zip(order, order[1:]) if a != b)
print("partition per message, 30 ms apart:", order)
print(f"busiest partition: {busiest}, holding {n_busiest} of 9"
      f"   ({dict(sorted(spread.items()))})")
print(f"switches: {switches} out of {len(order) - 1}")

partition per message, 30 ms apart: [0, 2, 0, 1, 2, 0, 0, 2, 0]
busiest partition: 0, holding 5 of 9   ({0: 5, 1: 1, 2: 3})
switches: 7 out of 8


Now compare the two distributions — not the two orders. With a pause the nine are spread
across the three partitions; without it they pile up. And the spread is **not** the orderly
0,1,2,0,1,2 of the folklore, because the choice is random, not rotating: two consecutive
messages can land together by chance, and it is only over many messages that it evens out.

The practical reading is worth more than the mechanism. **A fast producer without keys
concentrates; a slow one spreads.** So a topic that looks balanced in a test with a sleep
in the loop can arrive skewed in production, where the same code runs flat out — and the
imbalance is not in your data, it is in your throughput.

Two more things these cells show. The partition and the offset are **not** chosen by the
producer: they come back from the broker, because they are decided when the message is
appended to the log. And the delivery callbacks arrive **grouped by batch**, which is why
both cells carry the production index through the callback instead of trusting the order
the client hands back — reading that order as the production order is a mistake that is
easy to make and hard to notice.

---
## 3) With a key, the partition is a function of the key

The same events, now three elevator units with three events each — and this time
`unitId` is handed to `produce()` as the **key**. Same payload as a moment ago; the only
change is that the client is now told which field matters.

**Predict first:** will each unit's three messages land together? Will three keys
land in three different partitions? And what would change if the topic had 6
partitions instead of 3?

In [6]:
seen.clear()
for unit in ("EU-001", "EU-002", "EU-003"):
    for i in range(3):
        producer.produce(TOPIC, key=unit, value=door_event(unit, i),
                         on_delivery=lambda err, msg, i=i: report(err, msg, i))
producer.flush()

by_key = collections.defaultdict(set)
for i, key, val, part, off in seen:
    by_key[key.decode()].add(part)

for key in sorted(by_key):
    print(f"{key} -> partition(s) {sorted(by_key[key])}")

EU-001 -> partition(s) [1]
EU-002 -> partition(s) [1]
EU-003 -> partition(s) [2]


**A key never spreads.** Its partition is `hash(key) % partitions`, so every message
for `EU-001` is in one partition, in the order it was produced. That is the whole
reason keys exist: they are how you buy ordering for the things that need it, without
paying for a global order that nothing can scale.

And the relation is **N:1**, which is the answer to that line of the quiz: several
keys share a partition — with three keys and three partitions, two of them may well
have collided above — but one key is never split across two. Adding partitions later
changes `% partitions` and therefore moves keys: repartitioning an existing topic
breaks the ordering guarantee for the keys that move, which is why the partition count
is a decision you would rather get right once.

---
## 4) One consumer in a group reads every partition

A consumer joins a **group**. The group is what the broker tracks: which partitions
its members hold, and how far each has read.

**Predict:** one consumer, three partitions — how many of the 27 messages does it get?

In [7]:
def drain(consumers, seconds=10.0, quiet_for=2.0):
    """Poll every consumer until `quiet_for` seconds pass with nothing new.

    Returns {consumer name: [messages]}. Polling all of them in turn is what lets
    the group rebalance: a member that does not poll is a member that is not there.
    """
    got = {name: [] for name, _ in consumers}
    deadline, last = time.time() + seconds, time.time()
    while time.time() < deadline and time.time() - last < quiet_for:
        for name, c in consumers:
            msg = c.poll(0.2)
            if msg is None or msg.error():
                continue
            got[name].append(msg)
            last = time.time()
    return got

def group(name, gid):
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": gid,
                  "auto.offset.reset": "earliest"})
    c.subscribe([TOPIC])
    return (name, c)

solo = group("consumer-1", "dashboard")
got = drain([solo])
print("assigned partitions:", sorted(p.partition for p in solo[1].assignment()))
print("messages received  :", len(got["consumer-1"]))

assigned partitions: [0, 1, 2]
messages received  : 27


---
## 5) Two consumers in the same group split the partitions

A second consumer joins the **same** group. The broker rebalances.

**Predict:** how do three partitions divide between two consumers? And what happens
to a **fourth** consumer in a group with three partitions?

In [8]:
second = group("consumer-2", "dashboard")

# both must poll for the rebalance to complete; there is nothing to read, so the
# only thing this drain does is let the group settle
drain([solo, second], seconds=20.0, quiet_for=8.0)

for name, c in (solo, second):
    print(f"{name}: partitions {sorted(p.partition for p in c.assignment())}")

consumer-1: partitions [2]
consumer-2: partitions [0, 1]


Two and one, or one and two — which member gets two partitions is an accident of the
assignment strategy, and nothing in your design should depend on it.

What *is* guaranteed: **inside one group, a partition has exactly one reader.** So a
fourth consumer in this group would sit idle, holding nothing, consuming nothing — the
parallelism of a consumer group is capped by the partition count of the topic, which is
the same decision as in section 1, met again from the other end. Add it yourself and
look, it takes one line.

This is also the relation the quiz asks for: consumer to partition is **1:N** — one
consumer may hold several partitions, a partition is held by exactly one consumer.

---
## 6) A group is a cursor, not a queue

Reading does not consume. The messages are still in the log — that is what retention
is for. What a group holds is a **position**.

**Predict:** a brand-new group subscribes with `auto.offset.reset=earliest`. How many
of the 27 messages does it see?

In [9]:
audit = group("audit", "audit-trail")
got = drain([audit])
print("the new group received:", len(got["audit"]), "messages")
print()

print("where each group stands, as the BROKER remembers it:")
for name, c in (solo, second, audit):
    for tp in sorted(c.assignment(), key=lambda t: t.partition):
        lo, hi = c.get_watermark_offsets(tp, timeout=5)
        com = c.committed([tp], timeout=5)[0].offset
        com = "nothing committed" if com < 0 else f"committed {com}"
        print(f"  {name:10s} partition {tp.partition}: {com:20s} "
              f"log holds offsets {lo}..{hi - 1}")

the new group received: 27 messages

where each group stands, as the BROKER remembers it:
  consumer-1 partition 2: committed 6          log holds offsets 0..5
  consumer-2 partition 0: committed 14         log holds offsets 0..13
  consumer-2 partition 1: committed 7          log holds offsets 0..6
  audit      partition 0: nothing committed    log holds offsets 0..13
  audit      partition 1: nothing committed    log holds offsets 0..6
  audit      partition 2: nothing committed    log holds offsets 0..5


Read the *log holds* column first: the messages are **all still there**. Nothing was
removed by being read — they leave when retention says so, not when a consumer is done
with them.

The committed offset is the group's bookmark, and it lives on the broker rather than in
the consumer, which is what lets a crashed member be replaced by another that picks up
where it left off. A group may show *nothing committed*: committing is periodic and a
member that has just been through a rebalance may not have reached its next commit —
which is worth seeing too, because "the consumer read it" and "the group has recorded
that it read it" are different statements, and only the second one survives a crash.

Two groups, the same messages, two independent positions. This is why a Kafka topic
can feed a dashboard, a batch export and a machine-learning pipeline at once without
any of them knowing about the others — and why "the consumer is slow" is a question
about **lag**, the distance between a position and the end of the log, rather than
about a queue filling up.

It is also the mechanism behind `startingOffsets: "earliest"` in the Spark modules of
the next two lectures: Spark is a consumer group like any other.

---
## 7) The same event, in Avro

Five minutes, and it changes a number you are about to be asked for. The assignment at
the end of this lecture has you size a fleet of 150,000 units in MB/s, taking a JSON
event at about 250 bytes. **The encoding is a factor in that product**, and it is the
one you control without buying anything.

[Apache Avro](https://avro.apache.org/docs/) is a data serialization system: a schema
in JSON, and a compact binary encoding that carries **no field names at all** — the
reader knows them from the schema. The schema for our door event — the one you have been producing since section 2 — is
next to this notebook, in `door-event.avsc`. You exchanged it by hand, which is the honest way to
start: no registry, no infrastructure, a file.

In [10]:
import json

schema_str = open("door-event.avsc").read()
print(schema_str)

{
  "namespace": "it.polimi.sda",
  "type": "record",
  "name": "DoorEvent",
  "doc": "One door movement of one elevator. The naive translation of the JSON event: every field a string, exactly as it was written by hand.",
  "fields": [
    {"name": "event",     "type": "string"},
    {"name": "unitId",    "type": "string"},
    {"name": "ts",        "type": {"type": "long", "logicalType": "timestamp-millis"}},
    {"name": "car",       "type": "string"},
    {"name": "floor",     "type": "int"},
    {"name": "servedDir", "type": "string"}
  ]
}



**Predict:** the event you have been producing is 115 bytes as JSON. How small in Avro?

In [11]:
import io, datetime
from fastavro import parse_schema, schemaless_writer

event = {"event": "DoorOpened", "unitId": "EU-001",
         "ts": datetime.datetime(2026, 10, 15, 8, 0, 0,
                                 tzinfo=datetime.timezone.utc),
         "car": "A", "floor": 3, "servedDir": "UP"}

def avro_bytes(schema_dict, record):
    buf = io.BytesIO()
    schemaless_writer(buf, parse_schema(schema_dict), record)
    return buf.getvalue()

as_json = json.dumps({**event, "ts": "2026-10-15T08:00:00"}).encode()
as_avro = avro_bytes(json.loads(schema_str), event)

print("JSON               :", len(as_json), "bytes")
print("Avro               :", len(as_avro), "bytes ",
      f"-> {len(as_json)/len(as_avro):.1f}x smaller")
print("the schema itself  :", len(schema_str.encode()), "bytes, once, not per message")

JSON               : 115 bytes
Avro               : 30 bytes  -> 3.8x smaller
the schema itself  : 551 bytes, once, not per message


Now change **the schema and nothing else**. `event` and `servedDir` take one of a few
known values, so they are not strings, they are **enums** — and an enum on the wire is
an index, not a word.

**Predict again** before you run it.

In [12]:
better = json.loads(schema_str)
better["fields"][0]["type"] = {"type": "enum", "name": "EventType",
    "symbols": ["DoorOpened", "CarMoved", "HallCall", "Fault"]}
better["fields"][5]["type"] = {"type": "enum", "name": "Dir",
    "symbols": ["UP", "DOWN", "NONE"]}

as_enum = avro_bytes(better, event)
print("Avro, with enums   :", len(as_enum), "bytes ",
      f"-> {len(as_json)/len(as_enum):.1f}x smaller than JSON")

Avro, with enums   : 18 bytes  -> 6.4x smaller than JSON


The field names were already gone; now the *values* are gone too, and what is left on
the wire is almost only the floor number and the unit id. **Every byte you removed
moved into the schema**, which travels once.

Put that back into the assignment: a quarter or a sixth of the MB/s, for the same fleet,
the same rate and the same information. That is not a micro-optimisation, it is a
different cluster.

### Does it also mean more messages per batch?

It should, if a batch is bounded in **bytes**. The client will tell us: librdkafka
publishes statistics, and among them the average batch size, the average number of
messages per batch, and the bytes actually sent to the broker.

A thousand messages of each encoding, `linger.ms` at 50 so the batcher has something to
do — and each one twice: once with the default `batch.size`, once with it set to
**16 KB**, which is the default of the Java client. **Predict what changes between the
two, and what does not.**

In [13]:
for t, f in admin.create_topics([
        NewTopic("avro-demo-json", num_partitions=1, replication_factor=1),
        NewTopic("avro-demo-avro", num_partitions=1, replication_factor=1)]).items():
    f.result()
    print("created", t)

created avro-demo-json
created avro-demo-avro


In [14]:
def measure(topic, payload, batch_size=None, n=1000):
    """Produce n copies of `payload` and read the client's own statistics."""
    stats = {}
    conf = {"bootstrap.servers": BOOTSTRAP,
            "linger.ms": 50,
            "statistics.interval.ms": 500,
            "stats_cb": lambda raw: stats.update(json.loads(raw))}
    if batch_size:
        conf["batch.size"] = batch_size
    p = Producer(conf)
    for _ in range(n):
        p.produce(topic, value=payload)
    p.flush()
    deadline = time.time() + 5          # wait for one statistics sample
    while not stats and time.time() < deadline:
        p.poll(0.2)
    top = stats.get("topics", {}).get(topic, {})
    return (len(payload),
            sum(b.get("txbytes", 0) for b in stats.get("brokers", {}).values()),
            top.get("batchsize", {}).get("avg"),
            top.get("batchcnt", {}).get("avg"))

def fmt(v):
    return "n/a" if v is None else str(int(v))

head = ("encoding / setting", "payload", "wire bytes", "batch bytes", "msg/batch")
print("%-24s %8s %11s %12s %10s" % head)
row = {}
for enc, topic, payload in (("JSON", "avro-demo-json", as_json),
                            ("Avro", "avro-demo-avro", as_enum)):
    for bs, label in ((None, "batch.size default"), (16384, "batch.size 16 KB")):
        size, wire, b_bytes, b_msgs = measure(topic, payload, bs)
        row[(enc, bs)] = (size, wire, b_bytes, b_msgs)
        print("%-24s %8d %11d %12s %10s"
              % (enc + " " + label, size, wire, fmt(b_bytes), fmt(b_msgs)))

# the two readings that hold from run to run, said by the run itself
N = 1000                                   # the n of measure()
j, a = row[("JSON", None)], row[("Avro", None)]
print()
print(f"payload {j[0] / a[0]:.1f}x smaller, bytes on the wire only {j[1] / a[1]:.1f}x smaller")
print(f"  beyond the payload, per record: JSON {(j[1] - N * j[0]) / N:.1f} B, "
      f"Avro {(a[1] - N * a[0]) / N:.1f} B — the envelope, which no encoding removes")
j16, a16 = row[("JSON", 16384)][2], row[("Avro", 16384)][2]
if j16 and a16:
    print(f"at batch.size 16 KB: JSON fills {100 * j16 / 16384:.0f}% of the ceiling, "
          f"Avro {100 * a16 / 16384:.0f}% — only one of the two is limited by the setting")

encoding / setting        payload  wire bytes  batch bytes  msg/batch
JSON batch.size default       115      125297        41668        333
JSON batch.size 16 KB         115      125540        15622        125
Avro batch.size default        18       26281         8663        333
Avro batch.size 16 KB          18       26342         6499        250

payload 6.4x smaller, bytes on the wire only 4.8x smaller
  beyond the payload, per record: JSON 10.3 B, Avro 8.3 B — the envelope, which no encoding removes
at batch.size 16 KB: JSON fills 95% of the ceiling, Avro 40% — only one of the two is limited by the setting


Two readings hold every time you run this, and a third does not. Take them in that order.

**The bytes fall, but by less than the payload did.** A thousand events of 115 bytes put
about 125 KB on the wire, a thousand of 18 bytes about 26 KB: the payload shrank 6.4x and
the traffic only about 4.8x. The difference is **roughly ten bytes per record** that belong
to the batch format — offsets, timestamps, lengths — and no encoding of yours can remove
them. Below a certain size the envelope costs more than the letter.

**At `batch.size` 16 KB only one of the two encodings is limited by the setting.** JSON's
batches stop just under the ceiling; Avro's sit well below it, with room to spare. So the
honest description is not that the smaller payload fills the same batch better — it is
that **the smaller payload never reaches the ceiling at all**, while the larger one keeps
hitting it. Your encoding decides whether your own batching limit is a limit. That is the
sentence the cell prints for you, with this run's percentages.

**And the messages-per-batch column is not evidence.** Run the cell twice and it moves —
the run above shows 333 and 250, and the run before it showed 142 and 166 for the same
    two lines, with nothing changed in between. The
reason is in the first reading: with the default `batch.size` of a megabyte, a thousand
records of either size come nowhere near full, so what closes a batch is not its size but
the batcher waking up, and how often it wakes is a property of your laptop that afternoon.
A number that moves under you is not a measurement — it is worth knowing which of the
numbers a tool prints are of that kind.

And note what did *not* move: the bytes on the wire are the same under both `batch.size`
settings. Batching changes the number of requests, not the volume — which is why it is a
latency and CPU knob, and the encoding is the bandwidth one.

*(If a figure comes back as `None`, your build of librdkafka does not publish that
particular counter; the bytes are always there.)*

### And now the part this module does not teach

You exchanged the schema **by hand**, and for one producer and one consumer in one
notebook that is entirely fine. Now count what you would have to keep in step in a real
system: every producer, every consumer, every replay of old data, and next week a field
is added. Whose copy of `door-event.avsc` is the truth?

Sending the schema **with** each message answers it and undoes everything you just
measured. The other answer is a **schema registry**: a service that holds the schemas,
hands each one an id, and lets the message carry the id instead — plus a rule about
which changes are allowed, so that a new producer cannot break an old consumer.

That is also why the Avro support inside `confluent-kafka` — `AvroSerializer` — takes a
registry client as its **first argument**: it is built for that world, not for this cell.
Which of the two you need is an architectural decision, and now you have the numbers to
make it.

* [Apache Avro documentation](https://avro.apache.org/docs/) — the schema language and
  the binary encoding
* [Confluent Schema Registry](https://docs.confluent.io/platform/current/schema-registry/index.html)
  — *"a centralized repository for managing and validating schemas for topic message
  data, and for serialization and deserialization of the data over the network"*

---
## Clean up

In [15]:
for name, c in (solo, second, audit):
    c.close()

topics = [TOPIC, "avro-demo-json", "avro-demo-avro"]
for t, f in admin.delete_topics(topics, operation_timeout=30).items():
    f.result()
    print("deleted", t)

deleted elevator-doors
deleted avro-demo-json
deleted avro-demo-avro
